# Laboratory Exercise 8: Fine-Tuning of the Two Best-Performing Models

**Name:** Lorenzo Bela, Robert Callorina, Kean Guzon

**Section:** 58036

**Date:** 05/04/2026

**Dataset name:** Lab04 EDA Bias Dataset (bottled water, canned goods, combo, Noodles, Rice)

**Selected models from Lab 7:** MobileNetV2 and EfficientDetLite0

**Task type:** Image classification for both models

## Part A) Project Setup

In [ ]:
%pip install torch torchvision scikit-learn matplotlib seaborn pandas pillow timm -q

import os
import json
import shutil
import time
import random
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision.models import mobilenet_v2
import timm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from IPython.display import display, Markdown

plt.style.use("seaborn-v0_8-whitegrid")

NAME = "Lorenzo Bela, Robert Callorina, Kean Guzon"
SECTION = "58036"
DATE = "05/04/2026"
DATASET_NAME = "Lab04 EDA Bias Dataset (bottled water, canned goods, combo, Noodles, Rice)"
TOP_MODEL_1_NAME = "MobileNetV2"
TOP_MODEL_2_NAME = "EfficientDetLite0"
TOP_MODEL_1_TASK = "classification"
TOP_MODEL_2_TASK = "classification"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Name: {NAME}")
print(f"Section: {SECTION}")
print(f"Date: {DATE}")
print(f"Dataset name: {DATASET_NAME}")
print(f"Selected models: {TOP_MODEL_1_NAME} ({TOP_MODEL_1_TASK}), {TOP_MODEL_2_NAME} ({TOP_MODEL_2_TASK})")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    COLAB = True
except Exception:
    COLAB = False


def select_best_device() -> tuple[torch.device, dict[str, Any]]:
    info: dict[str, Any] = {
        "torch_cuda_available": bool(torch.cuda.is_available()),
        "torch_cuda_device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "colab": COLAB,
    }

    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        best_idx = 0
        best_mem = 0
        gpu_list: list[dict[str, Any]] = []
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            mem = int(props.total_memory)
            gpu_list.append({"index": idx, "name": props.name, "total_memory_gb": round(mem / (1024 ** 3), 2)})
            if mem > best_mem:
                best_mem = mem
                best_idx = idx
        torch.cuda.set_device(best_idx)
        selected = torch.device(f"cuda:{best_idx}")
        info["selected"] = gpu_list[best_idx]
        info["torch_cuda_devices"] = gpu_list
        return selected, info

    info["selected"] = {"name": "cpu"}
    info["warning"] = "CUDA was not detected. Training will fall back to CPU unless you enable a GPU runtime."
    return torch.device("cpu"), info


def find_workspace_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidates.extend([
        Path("/content/drive/MyDrive/Elective Machine Learning"),
        Path("/content/Elective Machine Learning"),
    ])
    for candidate in candidates:
        if (candidate / "ml-perception-labs").exists():
            return candidate
    return Path.cwd().resolve()


WORKSPACE_ROOT = find_workspace_root()
LABS_ROOT = WORKSPACE_ROOT / "ml-perception-labs"
PROJECT_ROOT = LABS_ROOT / "lab08_finetuning"
LAB7_ROOT = LABS_ROOT / "lab07_model_comparison"
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
LOGS_DIR = PROJECT_ROOT / "tuning_logs"
MODELS_DIR = PROJECT_ROOT / "finetuned_models"

for directory in [DATA_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if COLAB:
    drive.mount("/content/drive")
    print("Mounted Google Drive")

DEVICE, DEVICE_INFO = select_best_device()
print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"Selected GPU: {DEVICE_INFO.get('selected', {}).get('name', 'GPU')}")
    torch.backends.cudnn.benchmark = True
else:
    print(DEVICE_INFO.get("warning", ""))

if DEVICE.type != "cuda":
    raise RuntimeError("CUDA GPU is required for training. Enable a GPU runtime before running the training cells.")

## Part B) Selection of the Two Best Models from Lab 7

| Rank | Model Architecture / Name | Task Type | Lab 7 Performance | Justification for Inclusion |
| --- | --- | --- | --- | --- |
| Top 1 | MobileNetV2 | Classification | Test Accuracy: 0.5789, Best Val Accuracy: 0.8108, F1 (macro): 0.5429, Training Time: 513.9 s, Parameters: 2,230,277 | Highest test accuracy and macro F1 in Lab 7, strongest validation result, no overfitting flag, and the smallest parameter count among the top-performing models. |
| Top 2 | EfficientDetLite0 | Classification | Test Accuracy: 0.5263, Best Val Accuracy: 0.6216, F1 (macro): 0.4768, Training Time: 554.6 s, Parameters: 3,377,413 | Second-best test accuracy and competitive macro metrics, making it the strongest remaining candidate for fine-tuning after MobileNetV2. |

In [ ]:
LAB7_SUMMARY_PATH = LAB7_ROOT / "outputs" / "tables" / "lab07_model_comparison.csv"
if not LAB7_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Missing Lab 7 summary table: {LAB7_SUMMARY_PATH}")

lab7_df = pd.read_csv(LAB7_SUMMARY_PATH)
lab7_df["Parameters (#)"] = lab7_df["Parameters (#)"].astype(str).str.replace(",", "", regex=False).astype(int)
lab7_df["Test Accuracy"] = lab7_df["Test Accuracy"].astype(float)
lab7_df["Best Val Accuracy"] = lab7_df["Best Val Accuracy"].astype(float)
lab7_df["Training Time (s)"] = lab7_df["Training Time (s)"].astype(float)

selected_df = lab7_df.sort_values(
    by=["Test Accuracy", "Best Val Accuracy", "Training Time (s)", "Parameters (#)"],
    ascending=[False, False, True, True],
).head(2).reset_index(drop=True)

selected_df["Task Type"] = "classification"
selected_df["Model"] = selected_df["Model"].astype(str)

display(Markdown("### Lab 7 selection check"))
display(selected_df[["Model", "Task Type", "Test Accuracy", "Precision (macro)", "Recall (macro)", "F1-Score (macro)", "Training Time (s)", "Parameters (#)", "Best Val Accuracy", "Overfitting (Y/N)"]])

selected_models = selected_df["Model"].tolist()
print("Selected models:", selected_models)
if selected_models != [TOP_MODEL_1_NAME, TOP_MODEL_2_NAME]:
    print("Warning: automatic ranking differs from the notebook header. Using the ranked results above.")

## Part C) Diagnosing the Baseline

MobileNetV2 and EfficientDetLite0 both showed the same dominant failure mode in Lab 7: confusion among visually similar categories, especially between 'bottled water' and 'canned goods' due to cylindrical shapes and reflective surfaces, or 'combo' which contains elements from other categories. The confusion matrix indicates that the model struggled most when background clutter and lighting variations obscured defining features, while classes like 'Noodles' and 'Rice' with distinct textures were easier to separate. The fine-tuning plan below targets that weakness by lowering the learning rate, adding stronger regularization and augmentation, and first updating only the classification head before allowing the full network to adapt.

In [ ]:
class MobileNetV2Classifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.net = mobilenet_v2(weights=None)
        in_features = self.net.classifier[1].in_features
        self.net.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class EfficientDetLite0Classifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.net = timm.create_model("efficientnet_lite0", pretrained=False, num_classes=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def find_existing_file(candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not locate one of the expected artifact files.")


MODEL_PATHS = {
    TOP_MODEL_1_NAME: [
        LAB7_ROOT / "models" / "model_b.pth",
        Path("/content/drive/MyDrive/Elective Machine Learning/ml-perception-labs/lab07_model_comparison/models/model_b.pth"),
    ],
    TOP_MODEL_2_NAME: [
        LAB7_ROOT / "models" / "model_a.pth",
        Path("/content/drive/MyDrive/Elective Machine Learning/ml-perception-labs/lab07_model_comparison/models/model_a.pth"),
    ],
}


def build_model(model_name: str, num_classes: int) -> nn.Module:
    if model_name == TOP_MODEL_1_NAME:
        return MobileNetV2Classifier(num_classes)
    if model_name == TOP_MODEL_2_NAME:
        return EfficientDetLite0Classifier(num_classes)
    raise ValueError(f"Unknown model name: {model_name}")


def load_baseline_model(model_name: str, num_classes: int) -> nn.Module:
    model = build_model(model_name, num_classes).to(DEVICE)
    weight_path = find_existing_file(MODEL_PATHS[model_name])
    state_dict = torch.load(weight_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"Loaded baseline weights for {model_name} from {weight_path}")
    return model


baseline_models = {
    TOP_MODEL_1_NAME: load_baseline_model(TOP_MODEL_1_NAME, NUM_CLASSES),
    TOP_MODEL_2_NAME: load_baseline_model(TOP_MODEL_2_NAME, NUM_CLASSES),
}

In [ ]:
IMG_SIZE = 64
BASE_BATCH_SIZE = 64
BASE_NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
BASE_PIN_MEMORY = DEVICE.type == "cuda"
BASE_SEED = SEED

TRAIN_INDICES = None
VAL_INDICES = None
TEST_INDICES = None
CLASS_NAMES = None
NUM_CLASSES = None

LAB7_DATA_DIR = LAB7_ROOT / "data" / "raw"
if not LAB7_DATA_DIR.exists():
    raise FileNotFoundError(f"Missing Lab 7 dataset directory: {LAB7_DATA_DIR}")

source_candidates = [
    LAB7_DATA_DIR,
    LAB7_ROOT.parent / "lab07_model_comparison" / "data" / "raw",
    Path("/content/drive/MyDrive/Elective Machine Learning/ml-perception-labs/lab07_model_comparison/data/raw"),
]
source_data_dir = next((path for path in source_candidates if path.exists()), None)
if source_data_dir is None:
    raise FileNotFoundError("Could not locate the Lab 7 dataset source directory.")

if DATA_DIR.exists():
    print(f"Using dataset copy at {DATA_DIR}")
else:
    shutil.copytree(source_data_dir, DATA_DIR, dirs_exist_ok=True)
    print(f"Copied dataset from {source_data_dir} to {DATA_DIR}")

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

light_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_dataset = ImageFolder(root=DATA_DIR, transform=eval_transform)
CLASS_NAMES = base_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

split_generator = torch.Generator().manual_seed(BASE_SEED)
all_indices = torch.randperm(len(base_dataset), generator=split_generator).tolist()
n_total = len(all_indices)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
TRAIN_INDICES = all_indices[:n_train]
VAL_INDICES = all_indices[n_train:n_train + n_val]
TEST_INDICES = all_indices[n_train + n_val:]


def build_loaders(use_augmented_training: bool, batch_size: int = BASE_BATCH_SIZE) -> tuple[DataLoader, DataLoader, DataLoader]:
    train_dataset = ImageFolder(root=DATA_DIR, transform=train_transform if use_augmented_training else light_train_transform)
    eval_dataset = ImageFolder(root=DATA_DIR, transform=eval_transform)

    train_subset = torch.utils.data.Subset(train_dataset, TRAIN_INDICES)
    val_subset = torch.utils.data.Subset(eval_dataset, VAL_INDICES)
    test_subset = torch.utils.data.Subset(eval_dataset, TEST_INDICES)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=BASE_NUM_WORKERS, pin_memory=BASE_PIN_MEMORY)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=BASE_NUM_WORKERS, pin_memory=BASE_PIN_MEMORY)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=BASE_NUM_WORKERS, pin_memory=BASE_PIN_MEMORY)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_loaders(use_augmented_training=True)
print(f"Train: {len(TRAIN_INDICES)} | Val: {len(VAL_INDICES)} | Test: {len(TEST_INDICES)}")

In [ ]:
@torch.no_grad()
def evaluate_classifier(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    all_labels: list[int] = []
    all_preds: list[int] = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())

    test_acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return {
        "accuracy": float(test_acc),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "loss": float(total_loss / len(loader.dataset)),
    }


for model_name, model in baseline_models.items():
    metrics = evaluate_classifier(model, test_loader)
    print(f"Baseline {model_name}: {metrics}")

In [ ]:
def plot_failure_cases(model_name: str, model: nn.Module, loader: DataLoader, num_cases: int = 6) -> None:
    model.eval()
    failures = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            preds = logits.argmax(dim=1).cpu()
            labels = labels.cpu()
            for i in range(len(preds)):
                if preds[i] != labels[i]:
                    failures.append((images[i].cpu(), preds[i].item(), labels[i].item()))
                if len(failures) >= num_cases:
                    break
            if len(failures) >= num_cases:
                break
    
    if not failures:
        print(f"No failure cases found for {model_name}.")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for i, (img, pred, true) in enumerate(failures):
        # Unnormalize
        img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        img = img.clamp(0, 1).permute(1, 2, 0).numpy()
        axes[i].imshow(img)
        axes[i].set_title(f"True: {CLASS_NAMES[true]}\nPred: {CLASS_NAMES[pred]}", color='red')
        axes[i].axis('off')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
        
    plt.suptitle(f"{model_name} - Baseline Failure Cases")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"lab08_{model_name.lower()}_baseline_failures.png", dpi=150)
    plt.show()
    print(f"Saved {model_name} baseline failures to {FIGURES_DIR / f'lab08_{model_name.lower()}_baseline_failures.png'}")

for model_name, model in baseline_models.items():
    plot_failure_cases(model_name, model, test_loader, num_cases=6)

## Part D) Fine-Tuning Plan

| Hyperparameter / Setting | Top Model #1 | Top Model #2 |
| --- | --- | --- |
| Baseline value (from Lab 7) | Trained from scratch, no pretrained weights, LR = 1e-3, 30 epochs, Adam, batch size = 64, standard augmentation | Trained from scratch, no pretrained weights, LR = 1e-3, 30 epochs, Adam, batch size = 64, standard augmentation |
| New tuned value | Start from Lab 7 weights, LR = 1e-4 to 1e-5, 5-8 epochs, AdamW, stronger augmentation, weight decay, label smoothing | Start from Lab 7 weights, LR = 1e-4 to 1e-5, 5-8 epochs, AdamW, stronger augmentation, weight decay, label smoothing |
| Learning rate (and schedule) | 1e-4 or 3e-5 with cosine annealing / step decay | 1e-4 or 3e-5 with cosine annealing / step decay |
| Optimizer | AdamW | AdamW |
| Batch size | 64 with gradient accumulation when needed | 64 with gradient accumulation when needed |
| Number of epochs / iterations | 5-8 epochs per trial | 5-8 epochs per trial |
| Layer-freezing strategy | Head-only first, then full fine-tuning | Head-only first, then full fine-tuning |
| Data augmentation strategy | Horizontal flip, rotation, color jitter | Horizontal flip, rotation, color jitter |
| Regularization (dropout, weight decay, etc.) | Dropout = 0.2, weight decay = 1e-4, label smoothing = 0.1 | Dropout = 0.2, weight decay = 1e-4, label smoothing = 0.1 |
| Task-specific loss function | Cross-Entropy with label smoothing | Cross-Entropy with label smoothing |
| Early-stopping criterion | Stop after 2-3 validation plateaus | Stop after 2-3 validation plateaus |

In [ ]:
TRIAL_PLANS = {
    TOP_MODEL_1_NAME: [
        {
            "trial_id": "m1_t1",
            "freeze_backbone": True,
            "learning_rate": 1e-4,
            "scheduler": "cosine",
            "batch_size": 64,
            "epochs": 5,
            "weight_decay": 1e-4,
            "label_smoothing": 0.1,
            "use_augmentation": True,
            "grad_accumulation_steps": 1,
            "patience": 2,
        },
        {
            "trial_id": "m1_t2",
            "freeze_backbone": False,
            "learning_rate": 3e-5,
            "scheduler": "step",
            "batch_size": 64,
            "epochs": 6,
            "weight_decay": 1e-4,
            "label_smoothing": 0.05,
            "use_augmentation": True,
            "grad_accumulation_steps": 2,
            "patience": 2,
        },
        {
            "trial_id": "m1_t3",
            "freeze_backbone": False,
            "learning_rate": 1e-5,
            "scheduler": "cosine",
            "batch_size": 64,
            "epochs": 8,
            "weight_decay": 5e-5,
            "label_smoothing": 0.1,
            "use_augmentation": False,
            "grad_accumulation_steps": 2,
            "patience": 3,
        },
    ],
    TOP_MODEL_2_NAME: [
        {
            "trial_id": "m2_t1",
            "freeze_backbone": True,
            "learning_rate": 1e-4,
            "scheduler": "cosine",
            "batch_size": 64,
            "epochs": 5,
            "weight_decay": 1e-4,
            "label_smoothing": 0.1,
            "use_augmentation": True,
            "grad_accumulation_steps": 1,
            "patience": 2,
        },
        {
            "trial_id": "m2_t2",
            "freeze_backbone": False,
            "learning_rate": 3e-5,
            "scheduler": "step",
            "batch_size": 64,
            "epochs": 6,
            "weight_decay": 1e-4,
            "label_smoothing": 0.05,
            "use_augmentation": True,
            "grad_accumulation_steps": 2,
            "patience": 2,
        },
        {
            "trial_id": "m2_t3",
            "freeze_backbone": False,
            "learning_rate": 1e-5,
            "scheduler": "cosine",
            "batch_size": 64,
            "epochs": 8,
            "weight_decay": 5e-5,
            "label_smoothing": 0.1,
            "use_augmentation": False,
            "grad_accumulation_steps": 2,
            "patience": 3,
        },
    ],
}

for model_name, plans in TRIAL_PLANS.items():
    print(f"{model_name}: {len(plans)} fine-tuning trials ready")

## Part E) Execute Fine-Tuning

The notebook runs three fine-tuning trials per model with GPU acceleration, mixed precision, gradient accumulation, and per-trial logs saved to `tuning_logs/`. The best checkpoint from each model is copied to `finetuned_models/`.

In [ ]:
def configure_trainability(model: nn.Module, freeze_backbone: bool) -> None:
    for _, param in model.named_parameters():
        param.requires_grad = True

    if freeze_backbone:
        for name, param in model.named_parameters():
            if "classifier" not in name and "head" not in name:
                param.requires_grad = False


def make_optimizer(model: nn.Module, learning_rate: float, weight_decay: float) -> optim.Optimizer:
    trainable_params = [param for param in model.parameters() if param.requires_grad]
    return optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)


def make_scheduler(optimizer: optim.Optimizer, scheduler_name: str | None, epochs: int):
    if scheduler_name is None:
        return None
    if scheduler_name == "step":
        return optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.5)
    if scheduler_name == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    raise ValueError(f"Unknown scheduler: {scheduler_name}")


@torch.no_grad()
def inference_time_per_sample_ms(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    batch = next(iter(loader))
    images = batch[0].to(DEVICE, non_blocking=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    _ = model(images)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return (elapsed / images.size(0)) * 1000.0


def save_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


def save_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)


def train_trial(model_name: str, trial_cfg: dict[str, Any]) -> dict[str, Any]:
    train_loader, val_loader, test_loader = build_loaders(
        use_augmented_training=trial_cfg["use_augmentation"],
        batch_size=trial_cfg["batch_size"],
    )

    model = load_baseline_model(model_name, NUM_CLASSES)
    configure_trainability(model, trial_cfg["freeze_backbone"])

    criterion = nn.CrossEntropyLoss(label_smoothing=trial_cfg.get("label_smoothing", 0.0))
    optimizer = make_optimizer(model, trial_cfg["learning_rate"], trial_cfg["weight_decay"])
    scheduler = make_scheduler(optimizer, trial_cfg.get("scheduler"), trial_cfg["epochs"])
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    grad_accum = int(trial_cfg.get("grad_accumulation_steps", 1))
    patience = int(trial_cfg.get("patience", 3))

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    history: dict[str, list[float]] = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_state = None
    best_val_acc = -1.0
    best_epoch = 0
    epochs_no_improve = 0
    trial_start = time.perf_counter()

    for epoch in range(1, trial_cfg["epochs"] + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0
        optimizer.zero_grad(set_to_none=True)

        for step, (images, labels) in enumerate(train_loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = model(images)
                loss = criterion(logits, labels) / grad_accum

            scaler.scale(loss).backward()

            if step % grad_accum == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * grad_accum * images.size(0)
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            running_total += labels.size(0)

        train_loss = running_loss / running_total
        train_acc = running_correct / running_total

        model.eval()
        val_loss_total = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    logits = model(images)
                    loss = criterion(logits, labels)
                val_loss_total += loss.item() * images.size(0)
                val_correct += (logits.argmax(dim=1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_total / val_total
        val_acc = val_correct / val_total

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))
        history["train_acc"].append(float(train_acc))
        history["val_acc"].append(float(val_acc))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(
            f"[{model_name} | {trial_cfg['trial_id']}] Epoch {epoch:02d} | "
            f"TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | "
            f"ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}"
        )

        if epochs_no_improve >= patience:
            print(f"[{model_name} | {trial_cfg['trial_id']}] Early stopping at epoch {epoch}")
            break

    trial_time = time.perf_counter() - trial_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_classifier(model, test_loader)
    inf_ms = inference_time_per_sample_ms(model, test_loader)
    peak_memory_mb = float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if DEVICE.type == "cuda" else 0.0

    checkpoint_path = MODELS_DIR / f"{model_name}_{trial_cfg['trial_id']}.pth"
    torch.save(model.state_dict(), checkpoint_path)

    trial_log = {
        "model_name": model_name,
        "trial_id": trial_cfg["trial_id"],
        "config": trial_cfg,
        "best_epoch": best_epoch,
        "best_val_accuracy": float(best_val_acc),
        "test_accuracy": test_metrics["accuracy"],
        "test_precision_macro": test_metrics["precision_macro"],
        "test_recall_macro": test_metrics["recall_macro"],
        "test_f1_macro": test_metrics["f1_macro"],
        "training_time_s": float(trial_time),
        "inference_time_ms_per_sample": float(inf_ms),
        "peak_gpu_memory_mb": float(peak_memory_mb),
        "checkpoint_path": str(checkpoint_path),
        "history": history,
    }

    save_json(LOGS_DIR / f"{model_name}_{trial_cfg['trial_id']}.json", trial_log)
    save_csv(LOGS_DIR / f"{model_name}_{trial_cfg['trial_id']}.csv", [{
        "model_name": model_name,
        "trial_id": trial_cfg["trial_id"],
        "best_epoch": best_epoch,
        "best_val_accuracy": float(best_val_acc),
        "test_accuracy": test_metrics["accuracy"],
        "training_time_s": float(trial_time),
        "inference_time_ms_per_sample": float(inf_ms),
        "peak_gpu_memory_mb": float(peak_memory_mb),
    }])

    if COLAB:
        drive_root = Path("/content/drive/MyDrive/Elective Machine Learning/ml-perception-labs/lab08_finetuning")
        drive_root.mkdir(parents=True, exist_ok=True)
        for folder in ["tuning_logs", "finetuned_models", "outputs/tables", "outputs/figures"]:
            target_dir = drive_root / folder
            target_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(checkpoint_path, drive_root / "finetuned_models" / checkpoint_path.name)
        shutil.copy2(LOGS_DIR / f"{model_name}_{trial_cfg['trial_id']}.json", drive_root / "tuning_logs" / f"{model_name}_{trial_cfg['trial_id']}.json")
        shutil.copy2(LOGS_DIR / f"{model_name}_{trial_cfg['trial_id']}.csv", drive_root / "tuning_logs" / f"{model_name}_{trial_cfg['trial_id']}.csv")

    print(
        f"[{model_name} | {trial_cfg['trial_id']}] Best Val Acc={best_val_acc:.4f} | "
        f"Test Acc={test_metrics['accuracy']:.4f} | Time={trial_time:.1f}s | Peak GPU={peak_memory_mb:.1f} MB"
    )
    return trial_log

In [ ]:
trial_results: dict[str, list[dict[str, Any]]] = {TOP_MODEL_1_NAME: [], TOP_MODEL_2_NAME: []}
best_trial_results: dict[str, dict[str, Any]] = {}

for model_name in [TOP_MODEL_1_NAME, TOP_MODEL_2_NAME]:
    print("\n" + "=" * 80)
    print(f"Running fine-tuning trials for {model_name}")
    print("=" * 80)
    for trial_cfg in TRIAL_PLANS[model_name]:
        result = train_trial(model_name, trial_cfg)
        trial_results[model_name].append(result)
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    best_trial = max(trial_results[model_name], key=lambda item: item["best_val_accuracy"])
    best_trial_results[model_name] = best_trial
    best_checkpoint = MODELS_DIR / f"{model_name}_best.pth"
    shutil.copy2(best_trial["checkpoint_path"], best_checkpoint)
    print(f"Best trial for {model_name}: {best_trial['trial_id']} -> {best_checkpoint}")

trial_summary_rows = []
for model_name, results in trial_results.items():
    for result in results:
        trial_summary_rows.append({
            "model_name": model_name,
            "trial_id": result["trial_id"],
            "best_epoch": result["best_epoch"],
            "best_val_accuracy": result["best_val_accuracy"],
            "test_accuracy": result["test_accuracy"],
            "training_time_s": result["training_time_s"],
            "inference_time_ms_per_sample": result["inference_time_ms_per_sample"],
            "peak_gpu_memory_mb": result["peak_gpu_memory_mb"],
        })

trial_summary_df = pd.DataFrame(trial_summary_rows)
trial_summary_df.to_csv(LOGS_DIR / "lab08_trial_summary.csv", index=False)
display(trial_summary_df)
print(f"Saved trial summary: {LOGS_DIR / 'lab08_trial_summary.csv'}")

## Part F) Evaluate the Fine-Tuned Models

After the best trial is selected for each model, the notebook evaluates the fine-tuned checkpoint on the held-out test set, saves the confusion matrix, and exports the comparison table and grouped bar chart.

In [ ]:
def collect_predictions(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    labels_list: list[np.ndarray] = []
    preds_list: list[np.ndarray] = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            labels_list.append(labels.numpy())
            preds_list.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(labels_list), np.concatenate(preds_list)


def plot_training_curves(model_name: str, history: dict[str, list[float]]) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set_title(f"{model_name} Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="Train")
    axes[1].plot(epochs, history["val_acc"], label="Validation")
    axes[1].set_title(f"{model_name} Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"lab08_{model_name.lower()}_training_curves.png", dpi=150)
    plt.show()


baseline_metrics: dict[str, dict[str, float]] = {}
finetuned_metrics: dict[str, dict[str, float]] = {}

for model_name in [TOP_MODEL_1_NAME, TOP_MODEL_2_NAME]:
    baseline_metrics[model_name] = evaluate_classifier(baseline_models[model_name], test_loader)
    best_result = best_trial_results[model_name]
    best_model = build_model(model_name, NUM_CLASSES).to(DEVICE)
    best_model.load_state_dict(torch.load(best_result["checkpoint_path"], map_location=DEVICE))
    best_model.eval()
    finetuned_metrics[model_name] = evaluate_classifier(best_model, test_loader)
    y_true, y_pred = collect_predictions(best_model, test_loader)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"{model_name} - Fine-Tuned Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"lab08_{model_name.lower()}_confusion_matrix.png", dpi=150)
    plt.show()

    print(f"\n{model_name} classification report (fine-tuned):")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    plot_training_curves(model_name, best_result["history"])



In [ ]:
def plot_qualitative_comparison(model_name: str, baseline_model: nn.Module, tuned_model: nn.Module, loader: DataLoader, num_cases: int = 6) -> None:
    baseline_model.eval()
    tuned_model.eval()
    cases = []
    with torch.no_grad():
        for images, labels in loader:
            images_gpu = images.to(DEVICE, non_blocking=True)
            base_preds = baseline_model(images_gpu).argmax(dim=1).cpu()
            tuned_preds = tuned_model(images_gpu).argmax(dim=1).cpu()
            for i in range(len(images)):
                cases.append((images[i], labels[i].item(), base_preds[i].item(), tuned_preds[i].item()))
                if len(cases) >= num_cases:
                    break
            if len(cases) >= num_cases:
                break
                
    fig, axes = plt.subplots(num_cases, 2, figsize=(10, 4 * num_cases))
    if num_cases == 1:
        axes = [axes]
        
    for i, (img, true_lbl, base_pred, tuned_pred) in enumerate(cases):
        # Unnormalize
        img_disp = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        img_disp = img_disp.clamp(0, 1).permute(1, 2, 0).numpy()
        
        axes[i][0].imshow(img_disp)
        base_color = 'green' if base_pred == true_lbl else 'red'
        axes[i][0].set_title(f"Baseline Pred: {CLASS_NAMES[base_pred]}\n(True: {CLASS_NAMES[true_lbl]})", color=base_color)
        axes[i][0].axis('off')
        
        axes[i][1].imshow(img_disp)
        tuned_color = 'green' if tuned_pred == true_lbl else 'red'
        axes[i][1].set_title(f"Tuned Pred: {CLASS_NAMES[tuned_pred]}\n(True: {CLASS_NAMES[true_lbl]})", color=tuned_color)
        axes[i][1].axis('off')
        
    plt.suptitle(f"{model_name} - Qualitative Comparison (Before vs After)", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"lab08_{model_name.lower()}_qualitative_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved {model_name} qualitative comparison to {FIGURES_DIR / f'lab08_{model_name.lower()}_qualitative_comparison.png'}")

for model_name in [TOP_MODEL_1_NAME, TOP_MODEL_2_NAME]:
    best_result = best_trial_results[model_name]
    best_model = build_model(model_name, NUM_CLASSES).to(DEVICE)
    best_model.load_state_dict(torch.load(best_result["checkpoint_path"], map_location=DEVICE))
    best_model.eval()
    plot_qualitative_comparison(model_name, baseline_models[model_name], best_model, test_loader, num_cases=6)


## Part G) Generate the Comparison Summary

The code above writes `outputs/tables/lab08_finetuning_comparison.csv` and `outputs/figures/lab08_before_after_comparison.png`.

## Results and Discussion

Complete the written analysis after running the notebook:

- A. Diagnosis Summary
- B. Fine-Tuning Trials
- C. Quantitative Improvement
- D. Qualitative Improvement
- E. Computational Cost of Fine-Tuning
- F. Identified Limitations

## Questions and Conclusion

Answer the five questions in the lab instructions after the experiment finishes, then write the 8-10 sentence conclusion paragraph using the saved comparison table and figures.

In [ ]:
comparison_rows = []
for model_name in [TOP_MODEL_1_NAME, TOP_MODEL_2_NAME]:
    baseline = baseline_metrics[model_name]
    tuned = finetuned_metrics[model_name]
    best_result = best_trial_results[model_name]
    baseline_time = float(lab7_df.loc[lab7_df["Model"] == model_name, "Training Time (s)"].iloc[0])
    baseline_params = int(lab7_df.loc[lab7_df["Model"] == model_name, "Parameters (#)"].iloc[0])
    tuned_params = sum(param.numel() for param in build_model(model_name, NUM_CLASSES).parameters())
    best_val_loss = float(min(best_result["history"]["val_loss"]))
    train_val_gap = float(best_result["history"]["train_acc"][-1] - best_result["history"]["val_acc"][-1])
    improvement_abs = tuned["accuracy"] - baseline["accuracy"]
    improvement_pct = (improvement_abs / baseline["accuracy"] * 100.0) if baseline["accuracy"] else 0.0

    comparison_rows.append({
        "Metric": "Primary task metric (accuracy)",
        f"{model_name} (Before)": round(baseline["accuracy"], 4),
        f"{model_name} (After)": round(tuned["accuracy"], 4),
        "Notes": "Test accuracy on the held-out test split",
    })
    comparison_rows.append({
        "Metric": "Secondary task metric (F1 macro)",
        f"{model_name} (Before)": round(baseline["f1_macro"], 4),
        f"{model_name} (After)": round(tuned["f1_macro"], 4),
        "Notes": "Macro-averaged F1-score",
    })
    comparison_rows.append({
        "Metric": "Validation loss (best)",
        f"{model_name} (Before)": float(lab7_df.loc[lab7_df["Model"] == model_name, "Best Val Accuracy"].iloc[0]),
        f"{model_name} (After)": round(best_val_loss, 4),
        "Notes": "Lab 7 uses best validation accuracy; Lab 8 stores best validation loss here",
    })
    comparison_rows.append({
        "Metric": "Train-Validation gap (overfitting indicator)",
        f"{model_name} (Before)": float("nan"),
        f"{model_name} (After)": round(train_val_gap, 4),
        "Notes": "Computed from final epoch train/val accuracy",
    })
    comparison_rows.append({
        "Metric": "Training time (s)",
        f"{model_name} (Before)": round(baseline_time, 2),
        f"{model_name} (After)": round(best_result["training_time_s"], 2),
        "Notes": "Measured wall-clock time",
    })
    comparison_rows.append({
        "Metric": "Inference time per sample (ms)",
        f"{model_name} (Before)": round(inference_time_per_sample_ms(baseline_models[model_name], test_loader), 4),
        f"{model_name} (After)": round(best_result["inference_time_ms_per_sample"], 4),
        "Notes": "Single-batch inference timing",
    })
    comparison_rows.append({
        "Metric": "Trainable parameter count",
        f"{model_name} (Before)": baseline_params,
        f"{model_name} (After)": tuned_params,
        "Notes": "Total parameters from the active model definition",
    })
    comparison_rows.append({
        "Metric": "Improvement vs. baseline (Δ, %)",
        f"{model_name} (Before)": round(improvement_abs, 4),
        f"{model_name} (After)": f"{improvement_pct:.2f}%",
        "Notes": "Absolute and relative accuracy gain",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TABLES_DIR / "lab08_finetuning_comparison.csv", index=False)
display(comparison_df)
print(f"Saved comparison table: {TABLES_DIR / 'lab08_finetuning_comparison.csv'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
metrics_to_plot = ["accuracy", "f1_macro"]
metric_titles = ["Primary Task Metric (Accuracy)", "Secondary Metric (F1 Macro)"]
for ax, metric, title in zip(axes, metrics_to_plot, metric_titles):
    labels = []
    before_values = []
    after_values = []
    for model_name in [TOP_MODEL_1_NAME, TOP_MODEL_2_NAME]:
        labels.append(model_name)
        before_values.append(baseline_metrics[model_name][metric])
        after_values.append(finetuned_metrics[model_name][metric])

    x = np.arange(len(labels))
    width = 0.35
    bars_before = ax.bar(x - width / 2, before_values, width, label="Before", color="#C0392B")
    bars_after = ax.bar(x + width / 2, after_values, width, label="After", color="#27AE60")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.legend()
    ax.bar_label(bars_before, fmt="%.3f", padding=3)
    ax.bar_label(bars_after, fmt="%.3f", padding=3)

fig.suptitle("Lab 08 Before-vs-After Comparison", fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "lab08_before_after_comparison.png", dpi=150)
plt.show()
print(f"Saved grouped bar chart: {FIGURES_DIR / 'lab08_before_after_comparison.png'}")

if COLAB:
    drive_root = Path("/content/drive/MyDrive/Elective Machine Learning/ml-perception-labs/lab08_finetuning")
    shutil.copy2(TABLES_DIR / "lab08_finetuning_comparison.csv", drive_root / "outputs" / "tables" / "lab08_finetuning_comparison.csv")
    shutil.copy2(FIGURES_DIR / "lab08_before_after_comparison.png", drive_root / "outputs" / "figures" / "lab08_before_after_comparison.png")

print("Fine-tuning workflow complete.")